# Rajasthani Dialect AI - End-to-End Demo

This notebook demonstrates the Speech-to-Speech Translation (S2ST) pipeline for Rajasthani dialects.
It connects:
1. **ASR**: Whisper fine-tuned on dialect data (Speech → Devanagari text)
2. **MT**: IndicTrans2 with experience replay (Rajasthani/Hindi → English)
3. **TTS**: Bhashini API fallback (English/Hindi → Speech)

In [ ]:
import io
import base64
import numpy as np
import IPython.display as ipd
from src.asr.model import WhisperASR
from src.mt.model import IndicTrans2MT
from src.tts.fastpitch import IndicTTS

## 1. Load Models

For this demo, we'll initialize the skeleton/mock modes if HuggingFace weights aren't downloaded.

In [ ]:
print("Loading ASR...")
asr_model = WhisperASR()
print("Loading MT...")
mt_model = IndicTrans2MT()
print("Loading TTS...")
tts_model = IndicTTS()

## 2. Mock Audio Input (Recording)

In a real app, you would use `ipywebrtc` or similar to record from the mic. We'll use a dummy waveform here.

In [ ]:
# Generate 3 seconds of 16kHz dummy audio
sr = 16000
audio_input = np.random.randn(3 * sr).astype(np.float32) * 0.1

print("Playing input audio (simulated)...")
ipd.Audio(audio_input, rate=sr)

## 3. The S2ST Pipeline

In [ ]:
# 1. Transcribe (ASR)
print("Running ASR...")
transcription = asr_model.transcribe_array(audio_input, sampling_rate=sr, language="hi")
print(f"Transcription: {transcription}")

# 2. Translate (MT)
print("\nRunning MT...")
translations = mt_model.translate([transcription], src_lang="hindi", tgt_lang="english")
translated_text = translations[0]
print(f"Translation: {translated_text}")

# 3. Synthesize (TTS)
print("\nRunning TTS...")
audio_output = tts_model.synthesize(translated_text)
print("Playing output audio...")
ipd.Audio(audio_output, rate=tts_model.sample_rate)